# A LoRA layer in NumPy

A pretrained linear layer computes $W_0 x$ for a frozen $d_\text{out} \times d_\text{in}$
matrix $W_0$. LoRA adds a trainable update of rank $r$: the adapted layer computes

$$
W_0 x + \frac{\alpha}{r} B A x, \qquad A \in \mathbb{R}^{r \times d_\text{in}}, \quad B \in \mathbb{R}^{d_\text{out} \times r} .
$$

You will write the forward and backward pass by hand, train adapters for two tasks on
one shared $W_0$, merge an adapter into a single matrix, and serve a batch that mixes the
two tasks. Inputs are the rows of `X`, so a batch of outputs is
`X @ W0.T + (alpha / r) * X @ A.T @ B.T`.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. Each task wants $W_0$ plus a change of rank 2 that the task does not
reveal.

In [ ]:
import numpy as np

d_out, d_in = 24, 16
rng = np.random.default_rng(41)
W0 = rng.normal(0, 1 / np.sqrt(d_in), (d_out, d_in))       # the pretrained weight, frozen


def make_task(seed):
    """300 inputs and the outputs a task wants: W0 plus a rank-2 change, and a little noise."""
    r = np.random.default_rng(seed)
    change = r.normal(0, 0.3, (d_out, 2)) @ r.normal(0, 0.3, (2, d_in))
    X = r.normal(0, 1, (300, d_in))
    Y = X @ (W0 + change).T + 0.02 * r.normal(0, 1, (300, d_out))
    return X, Y


def mse(Y_hat, Y):
    """Half the squared error of a row, averaged over the rows."""
    return 0.5 * np.mean(np.sum((Y_hat - Y) ** 2, axis=1))


X1, Y1 = make_task(1)                                       # task 1
X2, Y2 = make_task(2)                                       # task 2
print(f"loss of the frozen layer on task 1: {mse(X1 @ W0.T, Y1):.3f}")

## Part 1 (coding task): Forward and backward

Write `lora_forward(X, W0, A, B, alpha)`, which returns the adapted layer's outputs for the
rows of `X`, and `lora_backward(X, A, B, alpha, G)`, which returns the gradients
`(dA, dB)`, the derivatives of the loss in every entry of `A` and of `B`, given `G`, the
loss's derivative in every output. The rank $r$ is `A.shape[0]`. The check compares your
gradients with central differences.

In [ ]:
def lora_forward(X, W0, A, B, alpha):
    """Outputs of the adapted layer for the rows of X, shape (n, d_out)."""
    # TODO
    raise NotImplementedError


def lora_backward(X, A, B, alpha, G):
    """(dA, dB) for the loss, given G, its derivative in the outputs, of shape (n, d_out)."""
    # TODO
    raise NotImplementedError

In [ ]:
g = np.random.default_rng(7)
Xc, Yc = g.normal(0, 1, (5, d_in)), g.normal(0, 1, (5, d_out))
Ac, Bc = g.normal(0, 1, (3, d_in)), g.normal(0, 1, (d_out, 3))
alpha_c = 6.0                                                    # alpha / r = 2

out = lora_forward(Xc, W0, Ac, Bc, alpha_c)
assert np.shape(out) == (5, d_out), f"the outputs have shape {np.shape(out)}, and they should have shape (5, {d_out})"
assert np.allclose(out, Xc @ (W0 + 2.0 * Bc @ Ac).T), "wrong outputs: they should equal x times (W0 + (alpha / r) B A) transposed, row by row"


def numeric(f, M, eps=1e-6):
    num = np.zeros_like(M)
    for idx in np.ndindex(M.shape):
        keep = M[idx]
        M[idx] = keep + eps
        up = f()
        M[idx] = keep - eps
        down = f()
        M[idx] = keep
        num[idx] = (up - down) / (2 * eps)
    return num


loss = lambda: mse(lora_forward(Xc, W0, Ac, Bc, alpha_c), Yc)
G = (lora_forward(Xc, W0, Ac, Bc, alpha_c) - Yc) / len(Xc)
dA, dB = lora_backward(Xc, Ac, Bc, alpha_c, G)
assert np.shape(dA) == Ac.shape and np.shape(dB) == Bc.shape, (
    f"dA should have A's shape {Ac.shape} and dB B's shape {Bc.shape}, got {np.shape(dA)} and {np.shape(dB)}")
for name, mine, M in (("dA", dA, Ac), ("dB", dB, Bc)):
    num = numeric(loss, M)
    if not np.allclose(mine, num, atol=1e-5):
        ratio = np.sum(mine * num) / np.sum(num * num)
        extra = f" (it is {ratio:.2f} times the numeric gradient)" if np.allclose(mine, ratio * num, atol=1e-5) else ""
        raise AssertionError(f"{name} does not match the numeric gradient{extra}")
print("Part 1 passed")

## Part 2 (coding task): Train an adapter

Write `train_lora(X, Y, W0, r, alpha, steps, lr, seed)`. Initialize `A` with
`np.random.default_rng(seed).normal(0, 1 / np.sqrt(d_in), (r, d_in))` and `B` as LoRA does,
then take `steps` gradient steps on `mse(lora_forward(...), Y)`, each using all the rows,
updating only `A` and `B`. Return the trained `A` and `B` and the list of losses, one
measured before each step.

In [ ]:
def train_lora(X, Y, W0, r, alpha, steps, lr, seed):
    """Train a rank-r adapter on (X, Y) with W0 frozen.

    Return (A, B, losses): the trained factors and the loss before each step.
    """
    # TODO
    raise NotImplementedError

In [ ]:
W0_before = W0.copy()
A1, B1, losses1 = train_lora(X1, Y1, W0, r=4, alpha=8, steps=300, lr=0.1, seed=0)
assert np.array_equal(W0, W0_before), "W0 changed during training: only A and B should be updated"
assert np.shape(A1) == (4, d_in) and np.shape(B1) == (d_out, 4), (
    f"for r = 4 expected A of shape (4, {d_in}) and B of shape ({d_out}, 4), got {np.shape(A1)} and {np.shape(B1)}")
assert len(losses1) == 300, f"expected 300 losses, one before each step, got {len(losses1)}"
frozen = mse(X1 @ W0.T, Y1)
assert np.isclose(losses1[0], frozen), (
    f"the first loss is {losses1[0]:.4f}, but the frozen layer's loss is {frozen:.4f}: "
    "before any step the adapted layer should compute exactly what W0 computes")
assert not np.isclose(losses1[-1], losses1[0], rtol=0, atol=1e-9), (
    f"the loss stayed at {losses1[0]:.4f} for 300 steps: look at what each factor's gradient is at the start")
assert np.isfinite(losses1[-1]) and losses1[-1] < 0.01, (
    f"the loss went from {losses1[0]:.3f} to {losses1[-1]:.4f}: expected below 0.01 after 300 steps")
final = mse(lora_forward(X1, W0, A1, B1, 8), Y1)
assert np.isfinite(final) and final < 0.01, (
    f"the trained factors give a loss of {final:.4f}: expected below 0.01 after 300 steps")
rank = np.linalg.matrix_rank(B1 @ A1)
print(f"task 1: loss {losses1[0]:.3f} -> {losses1[-1]:.4f}, rank of B A = {rank}")
print(f"trainable numbers: {A1.size + B1.size} against {W0.size} in W0")
print("Part 2 passed")

## Part 3 (coding task): Merge, and serve two tasks at once

Write `merge(W0, A, B, alpha)`, the one matrix that computes what the adapted layer
computes. Then write `serve(X, task, W0, adapters, alpha)` for a batch whose rows belong
to different tasks: `adapters` is a list of `(A, B)` pairs, and row `i` of `X` must get
the output of task `task[i]`'s adapted layer. Do not change `W0`.

In [ ]:
def merge(W0, A, B, alpha):
    """The single weight matrix that computes what the adapted layer computes."""
    # TODO
    raise NotImplementedError


def serve(X, task, W0, adapters, alpha):
    """Outputs for a mixed batch: row i uses adapters[task[i]] on top of the shared W0."""
    # TODO
    raise NotImplementedError

In [ ]:
A2, B2, losses2 = train_lora(X2, Y2, W0, r=4, alpha=8, steps=300, lr=0.1, seed=1)
W1_merged = merge(W0, A1, B1, alpha=8)
assert np.shape(W1_merged) == W0.shape, f"the merged matrix should have W0's shape {W0.shape}, got {np.shape(W1_merged)}"
assert np.allclose(X1 @ W1_merged.T, lora_forward(X1, W0, A1, B1, 8)), (
    "the merged matrix does not reproduce the adapted layer: is the update scaled by alpha / r?")

g = np.random.default_rng(5)
task = g.integers(0, 2, 40)
Xmix = np.where(task[:, None] == 0, X1[:40], X2[:40])
W0_before = W0.copy()
out = serve(Xmix, task, W0, [(A1, B1), (A2, B2)], alpha=8)
assert np.array_equal(W0, W0_before), "serve changed W0, which every task shares"
want = np.where(task[:, None] == 0, Xmix @ merge(W0, A1, B1, 8).T, Xmix @ merge(W0, A2, B2, 8).T)
assert np.shape(out) == want.shape, f"serve returned shape {np.shape(out)} for a batch that needs {want.shape}"
bad = np.flatnonzero(~np.all(np.isclose(out, want), axis=1))
assert bad.size == 0, (
    f"rows {bad[:5].tolist()} (tasks {(task[bad[:5]] + 1).tolist()}) do not match the merged model of their task: "
    "task 1 is adapters[0] and task 2 is adapters[1]")
for t, (Xt, Yt) in enumerate(((X1, Y1), (X2, Y2))):
    rows = task == t
    print(f"task {t + 1} rows: loss {mse(out[rows], Yt[:40][rows]):.4f} served together, "
          f"{mse(Xt[:40][rows] @ W0.T, Yt[:40][rows]):.3f} with W0 alone")
print("Part 3 passed")

## Solution

### Part 1: Forward and backward

In [ ]:
def lora_forward(X, W0, A, B, alpha):
    """Outputs of the adapted layer for the rows of X, shape (n, d_out)."""
    s = alpha / A.shape[0]
    return X @ W0.T + s * (X @ A.T) @ B.T


def lora_backward(X, A, B, alpha, G):
    """(dA, dB) for the loss, given G, its derivative in the outputs, of shape (n, d_out)."""
    s = alpha / A.shape[0]
    dB = s * G.T @ (X @ A.T)            # (d_out, r): the error against what A passed up
    dA = s * (G @ B).T @ X              # (r, d_in): the error sent back through B, against the input
    return dA, dB

**Why it works.** Write $H = XA^\top$ for the $n \times r$ array the down-projection passes up, so the
adapted output is $XW_0^\top + sHB^\top$ with $s = \alpha / r$. For the term $sHB^\top$
the gradient of $B$ is $s\,G^\top H$, the outer products of each row's error with what
reached $B$. The gradient that reaches $H$ is $s\,GB$, and $H = XA^\top$ turns it into
$dA = s\,(GB)^\top X$. Each factor's gradient has the other factor in it, which is the
reason LoRA starts one factor at zero and not both. The check uses $\alpha / r = 2$, so a
missing scale shows up as half the true gradient.

### Part 2: Train an adapter

In [ ]:
def train_lora(X, Y, W0, r, alpha, steps, lr, seed):
    """Train a rank-r adapter on (X, Y) with W0 frozen.

    Return (A, B, losses): the trained factors and the loss before each step.
    """
    g = np.random.default_rng(seed)
    A = g.normal(0, 1 / np.sqrt(X.shape[1]), (r, X.shape[1]))
    B = np.zeros((W0.shape[0], r))                  # B starts at zero, so training starts from W0
    losses = []
    for _ in range(steps):
        Y_hat = lora_forward(X, W0, A, B, alpha)
        losses.append(mse(Y_hat, Y))
        dA, dB = lora_backward(X, A, B, alpha, (Y_hat - Y) / len(X))
        A = A - lr * dA
        B = B - lr * dB
    return A, B, losses

**Why it works.** With $B = 0$ the product $BA$ is zero, so the first forward pass is the frozen layer's
and the first loss equals its loss exactly. $B$'s first gradient, $s\,G^\top XA^\top$, is
not zero because $A$ is random, so $B$ moves on the first step and $A$ follows from the
second. Only $r(d_\text{in} + d_\text{out}) = 160$ numbers train against the 384 of $W_0$,
and $BA$ has rank at most 4, enough here because the task's change has rank 2. The loss
ends near $\tfrac12 \cdot 24 \cdot 0.02^2 = 0.0048$, the part of the loss that comes from the
noise in $Y$, which no weight can remove.

### Part 3: Merge, and serve two tasks at once

In [ ]:
def merge(W0, A, B, alpha):
    """The single weight matrix that computes what the adapted layer computes."""
    return W0 + (alpha / A.shape[0]) * B @ A


def serve(X, task, W0, adapters, alpha):
    """Outputs for a mixed batch: row i uses adapters[task[i]] on top of the shared W0."""
    task = np.asarray(task)
    out = X @ W0.T                                   # the shared path, once for every row
    for t, (A, B) in enumerate(adapters):
        rows = task == t
        out[rows] += (alpha / A.shape[0]) * (X[rows] @ A.T) @ B.T
    return out

**Why it works.** $W_0x + sB(Ax) = (W_0 + sBA)x$ for every input, so the merged matrix has $W_0$'s shape and
gives the same outputs with one product instead of three, and no cost at inference. The
mixed batch cannot use one merged matrix, because its rows want different ones. Instead
the base product $XW_0^\top$ runs once for every row, and each adapter adds its low-rank
term to its own rows. The two adapters together hold 320 numbers next to the 384 of
$W_0$ here. In a real model $W_0$ is far larger, which is why one base can serve many
tasks this way.